# Portfolios and strategies

A portfolio is a list of contracts. Its value is the sum of the legs. A strategy is the
shorter description that builds those contracts: a collar, a strip of forwards, a seagull,
or a participating forward. Pricing is the bank's book. `client` says whether the bank
faces a producer or a consumer. The sample book is the collar, the outright, and the Asian
from `examples.sample_contracts`.


In [1]:
import logging
import math

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import matplotlib.pyplot as plt
import numpy as np
import sjd
from sjd import examples, strategy

sjd.set_log_level("INFO")

market = examples.sample_market()
environment = market["environment"]
sjd.require_valid(environment)
book = examples.sample_contracts(market["forward"], market)
for contract in book["contracts"]:
    sjd.require_valid(contract, environment)
print("contracts", len(book["contracts"]))


contracts 4


## A portfolio is the sum of its legs

`contracts()` returns the instruments that were passed in. The sample book's value is
positive because the forward and the Asian are struck below the curve.


In [2]:
portfolio = sjd.Portfolio(book["contracts"])
sjd.require_valid(portfolio, environment)
leg_sum = sum(contract.get_value(environment) for contract in book["contracts"])

assert portfolio.contracts() == book["contracts"]
assert math.isclose(portfolio.get_value(environment), leg_sum)
assert portfolio.get_value(environment) > 0.0
assert "Portfolio" in repr(portfolio)
print(repr(portfolio))
print(f"portfolio {portfolio.get_value(environment):.2f}")
print(f"legs      {leg_sum:.2f}")


sjd.Portfolio(contracts=4)
portfolio 435989.15
legs      435989.15


## Strategies

`from_moneyness` sets each strike to `forward * moneyness`, rounded to two decimals.
`contracts()` builds the bank's instruments and `portfolio()` wraps them. The default
client is a producer, so a collar is a short put and a long call, a seagull adds a long
put at a lower strike, and a participating forward is a short put on the full volume and
a long call on half of it. `premium` is the bank's option value, with sold options
negative. A strip has no options, so its premium is zero. Buying the strip is the bank
buying from a producer. `merge_contracts` concatenates the lists.


In [ ]:
collar = strategy.sample_collar(market, market["forward"])
assert collar.put_strike == book["put_strike"]
assert collar.call_strike == book["call_strike"]
put, call = collar.contracts()
assert collar.client is strategy.Client.PRODUCER
assert put.buy_sell == sjd.BuySell.SELL
assert call.buy_sell == sjd.BuySell.BUY
sjd.require_valid(put, environment)
sjd.require_valid(call, environment)
collar_book = collar.portfolio()
sjd.require_valid(collar_book, environment)
assert math.isclose(
    collar_book.get_value(environment),
    put.get_value(environment) + call.get_value(environment),
)
assert math.isclose(
    collar.premium(environment),
    call.get_option_value(environment) - put.get_option_value(environment),
)
consumer = strategy.sample_collar(
    market, market["forward"], client=strategy.Client.CONSUMER
)
consumer_put, consumer_call = consumer.contracts()
assert consumer_put.buy_sell == sjd.BuySell.BUY
assert consumer_call.buy_sell == sjd.BuySell.SELL

atm = market["forward"].get_forward(market["collar_settlement"])
strip = strategy.StripStrategy.from_moneyness(
    [market["collar_settlement"], market["forward_settlement"]],
    market["forward"],
    0.98,
    book["volume"],
)
assert strip.strikes == tuple(
    round(market["forward"].get_forward(day) * 0.98, 2) for day in strip.settlements
)
assert strip.premium(environment) == 0.0
for contract in strip.contracts():
    sjd.require_valid(contract, environment)

seagull = strategy.SeagullStrategy.from_moneyness(
    market["collar_maturity"],
    market["collar_settlement"],
    atm,
    0.90,
    0.97,
    1.03,
    book["volume"],
)
assert seagull.floor_strike < seagull.put_strike < seagull.call_strike
for contract in seagull.contracts():
    sjd.require_valid(contract, environment)
seagull_put, seagull_call, seagull_floor = seagull.contracts()
assert seagull_put.buy_sell == sjd.BuySell.SELL
assert seagull_call.buy_sell == sjd.BuySell.BUY
assert seagull_floor.buy_sell == sjd.BuySell.BUY
assert math.isclose(
    seagull.premium(environment),
    -seagull_put.get_option_value(environment)
    + seagull_call.get_option_value(environment)
    + seagull_floor.get_option_value(environment),
)
assert len(seagull.contracts()) == 3

participating = strategy.ParticipatingForwardStrategy.from_moneyness(
    market["collar_maturity"],
    market["collar_settlement"],
    atm,
    0.97,
    1.03,
    1_000.0,
    0.5,
)
assert math.isclose(participating.call_volume(), 500.0)
assert participating.contracts()[1].volume == 500.0
for contract in participating.contracts():
    sjd.require_valid(contract, environment)

merged = strategy.merge_contracts(
    collar.contracts(), strip.contracts(), seagull.contracts(), participating.contracts()
)
assert len(merged) == 2 + len(strip.contracts()) + 3 + 2
print("collar premium", round(collar.premium(environment), 2))
print("seagull premium", round(seagull.premium(environment), 2))
print("merged", len(merged))


## Payoff diagrams

`get_payoff` is the settlement amount at a spot, before discounting, on the bank's book.
The collar is short the put and long the call. The seagull is also long the lower put, so
below that strike its payoff sits above the collar. The participating forward is short the
put on the full volume and long the call on half of it. The strip is two forwards the bank
has bought.


In [ ]:
spots = np.linspace(14.0, 22.0, 81)
diagrams = [
    ("Collar", collar.contracts()),
    ("Strip", strip.contracts()),
    ("Seagull", seagull.contracts()),
    ("Participating forward", participating.contracts()),
]
figure, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for axis, (title, contracts) in zip(axes.ravel(), diagrams):
    payoffs = [sum(contract.get_payoff(spot) for contract in contracts) for spot in spots]
    axis.plot(spots, payoffs)
    axis.axhline(0.0, color="0.7", linewidth=0.8)
    axis.set_title(title)
axes[1, 0].set_xlabel("Spot")
axes[1, 1].set_xlabel("Spot")
figure.tight_layout()

def payoff_at(contracts, spot):
    return sum(contract.get_payoff(spot) for contract in contracts)

assert math.isclose(
    payoff_at(collar.contracts(), 14.0),
    -(collar.put_strike - 14.0) * collar.volume,
)
assert math.isclose(payoff_at(collar.contracts(), collar.call_strike + 1.0), collar.volume)
assert payoff_at(seagull.contracts(), 14.0) > payoff_at(collar.contracts(), 14.0)
figure


## Margin

`SimpleMargin` turns a mark-to-market into a call. With both thresholds at 1,000, a minimum
transfer of 500, and rounding to 100, a mark-to-market of 10,000 calls 9,000, 1,400 calls
nothing, and -10,000 calls -9,000. `set_margin_parameters` stores an initial margin and does
not change that call. `Portfolio(contracts, margin)` is margined, and its call is the margin
rule applied to the portfolio value.


In [ ]:
margin = sjd.SimpleMargin(
    market["anchor"], True, 1000.0, True, 1000.0, 500.0, 100.0
)
sjd.require_valid(margin)
assert margin.get_margin_call(10_000.0) == 9_000.0
assert margin.get_margin_call(1_400.0) == 0.0
assert margin.get_margin_call(-10_000.0) == -9_000.0

margin.set_margin_parameters(
    market["anchor"], True, 1000.0, True, 250.0, True, 1000.0, False, 0.0, 500.0, 100.0
)
assert margin.requires_initial_margin()
assert margin.initial_margin() == 250.0
assert margin.get_margin_call(10_000.0) == 9_000.0

held = sjd.Portfolio(book["contracts"], margin)
sjd.require_valid(held, environment)
assert held.is_margined()
assert math.isclose(
    held.get_margin_call(environment),
    margin.get_margin_call(held.get_value(environment)),
)
held.set_margin_balance(1_234.5)
assert held.get_margin_balance() == 1_234.5
print(f"margin call {held.get_margin_call(environment):.2f}")
print(f"balance {held.get_margin_balance():.2f}")


## What can go wrong

A portfolio validated against an empty environment has one report. The path names the
portfolio, the contract index, and the instrument. A strategy refuses a put struck above
its call at construction. A negative initial margin fails `validate`.


In [ ]:
empty = sjd.RatesEnvironment(market["anchor"])
lone = sjd.EuropeanOption(
    market["collar_maturity"],
    market["collar_settlement"],
    book["call_strike"],
    sjd.PutCall.CALL,
    book["volume"],
    market["collar_settlement"],
    0.0,
    sjd.BuySell.BUY,
)
empty_book = sjd.Portfolio([lone])
empty_report = empty_book.validate(empty)
assert not empty_report.ok
assert any(
    issue.path[:3] == ("portfolio", "contracts[0]", "european_option")
    for issue in empty_report.issues
)

try:
    strategy.CollarStrategy(
        maturity=market["collar_maturity"],
        settlement=market["collar_settlement"],
        put_strike=20.0,
        call_strike=18.0,
        volume=1.0,
    )
except ValueError as error:
    assert "put_strike" in str(error)
    print(error)
else:
    raise AssertionError("a put above the call is refused")

broken_margin = sjd.SimpleMargin(
    market["anchor"], True, 1000.0, True, 1000.0, 500.0, 100.0
)
broken_margin.set_margin_parameters(
    market["anchor"], True, 1000.0, True, -1.0, True, 1000.0, False, 0.0, 500.0, 100.0
)
assert not broken_margin.validate().ok
empty_report


## What it logs

Building a strategy portfolio writes one `INFO` record on `sjd.strategy`. The message names
the strategy and the number of contracts.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []

    def emit(self, record):
        self.records.append(record)

captured = Collect()
logging.getLogger("sjd.strategy").addHandler(captured)
collar.portfolio()
strip.portfolio()
seagull.portfolio()
participating.portfolio()
messages = [record.getMessage() for record in captured.records]

assert any(message.startswith("built collar portfolio") for message in messages)
assert any(message.startswith("built strip portfolio") for message in messages)
assert any(message.startswith("built seagull portfolio") for message in messages)
assert any(message.startswith("built participating forward portfolio") for message in messages)
assert all(record.levelno == logging.INFO for record in captured.records)
for message in messages:
    print("INFO", message)


## Save and reload

A margined portfolio pickles with its margin balance. A strategy is a frozen dataclass of
dates and numbers: it pickles, and `dataclasses.asdict` is the plain record of its fields.
The file is in a temporary directory.


In [ ]:
import pickle
import tempfile
from dataclasses import asdict
from pathlib import Path

scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "portfolio.pkl"
path.write_bytes(pickle.dumps((held, collar)))
restored_portfolio, restored_collar = pickle.loads(path.read_bytes())

assert restored_portfolio.is_margined()
assert restored_portfolio.get_margin_balance() == 1_234.5
assert math.isclose(
    restored_portfolio.get_value(environment), held.get_value(environment)
)
assert restored_collar == collar
fields = asdict(collar)
assert fields["put_strike"] == collar.put_strike
assert fields["call_strike"] == collar.call_strike
assert fields["volume"] == collar.volume
print("reloaded", path.name)
print(fields)
